In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# ----------------------------
# Helpers
# ----------------------------
def encode_target(y_train, y_valid=None):
    """
    Convierte etiquetas de texto a enteros.
    Útil si tu target es algo como 'A', 'B', 'cat', 'dog', etc.
    """
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)

    y_valid_enc = le.transform(y_valid)
    return y_train_enc, y_valid_enc, le


def make_preprocessor(scale_numeric: bool = True) -> ColumnTransformer:
    """
    Preprocesador para tablas:
    - numéricas: imputación + (opcional) StandardScaler
    - categóricas: imputación + OneHotEncoder
    """
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, selector(dtype_include=np.number)),
            ("cat", cat_pipe, selector(dtype_include=["object", "category"])),
        ],
        remainder="drop",
    )


def fit_and_eval(model, X_train, y_train, X_valid, y_valid, average="weighted"):
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)

    acc = accuracy_score(y_valid, pred)
    f1 = f1_score(y_valid, pred, average=average, zero_division=0)

    print(f"accuracy: {acc:.4f}")
    print(f"f1 ({average}): {f1:.4f}")
    return model


# ----------------------------
# 1) Logistic Regression
# ----------------------------
def make_logistic_model():
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
        )),
    ])


# ----------------------------
# 2) Random Forest
# ----------------------------
def make_random_forest_model():
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        )),
    ])


# ----------------------------
# 3) XGBoost
# ----------------------------
def make_xgboost_model(y_train=None):
    params = dict(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method="hist",
        eval_metric="logloss",
    )

    # Si ya tienes y_train, ajustamos el objetivo para binaria vs multiclase
    if y_train is not None:
        n_classes = len(np.unique(y_train))
        if n_classes == 2:
            params["objective"] = "binary:logistic"
            params["eval_metric"] = "logloss"
        else:
            params["objective"] = "multi:softprob"
            params["eval_metric"] = "mlogloss"
            params["num_class"] = n_classes

    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", XGBClassifier(**params)),
    ])


# ----------------------------
# 4) SVC
# ----------------------------
def make_svc_model():
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
        )),
    ])


# ----------------------------
# 5) KNeighborsClassifier
# ----------------------------
def make_knn_model():
    return Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
        )),
    ])


# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    # df = pd.read_csv("train.csv")
    # target_col = "target"
    #
    # X = df.drop(columns=[target_col])
    # y = df[target_col]
    #
    # y_train, y_valid, le = encode_target(y_train, y_valid)
    #
    # Example split:
    # X_train, X_valid, y_train, y_valid = train_test_split(
    #     X, y, test_size=0.2, random_state=42, stratify=y
    # )
    #
    # If y is text, encode it:
    # y_train, y_valid, le = encode_target(y_train, y_valid)

    pass